# Tutorial 12: JWT Authentication and Authorization (Offline-First)

Focused tutorial for `A2A/12_JWT_authentication_authorizatoin`, covering token issuance, token validation, and protected execution flow.

## 1) Where We Are in the Journey

`11_authentication_authorizatoin_key_based` secured requests with static keys.
This step upgrades to token-based authorization with expiration semantics.
It exists to support stronger, session-aware trust in A2A controllers.

## 2) What You Will Learn

- Token issuance and verification flow
- Protected endpoint behavior with/without valid token
- Why JWT-like patterns scale better than static keys
- How the folder’s test notebook maps to auth-then-execute workflow

## 3) Why This Matters

Token auth enables scoped, expiring access with better security hygiene.
It is essential for distributed multi-service orchestration.
JWT-style patterns reduce risk compared with static credentials.

## 4) Core Concept Deep Dive

Source notebook calls `/execute` and `/result` flows where auth is expected in real deployments.
This tutorial demonstrates token creation and validation offline using signed payloads.
Trade-off: token systems add complexity (issuance, rotation, expiry) but improve security posture.

## 5) Code Walkthrough (Only `A2A/12_JWT_authentication_authorizatoin`)

- `test.ipynb` executes request, checks status and result paths.
- Tutorial keeps same execute/result usage intent and adds explicit auth simulation.
- Emphasis is on token-gated endpoint behavior under success and failure states.

In [ ]:
import base64
import hashlib
import hmac
import json
import time

SECRET = b'super-secret-key'

def issue_token(subject, ttl_seconds=60):
    payload = {'sub': subject, 'exp': int(time.time()) + ttl_seconds}
    data = json.dumps(payload, separators=(',', ':')).encode()
    sig = hmac.new(SECRET, data, hashlib.sha256).hexdigest().encode()
    return base64.urlsafe_b64encode(data + b'.' + sig).decode()

def verify_token(token):
    try:
        raw = base64.urlsafe_b64decode(token.encode())
        data, sig = raw.rsplit(b'.', 1)
        expected = hmac.new(SECRET, data, hashlib.sha256).hexdigest().encode()
        if not hmac.compare_digest(sig, expected):
            return False, 'bad_signature', None
        payload = json.loads(data.decode())
        if int(time.time()) > payload['exp']:
            return False, 'expired', payload
        return True, 'ok', payload
    except Exception:
        return False, 'invalid', None

In [ ]:
def protected_execute(query, token):
    ok, reason, payload = verify_token(token)
    if not ok:
        return {'status': 'error', 'code': 401, 'reason': reason}
    # Simulated business logic
    if 'add 10 and 20' in query:
        return {'status': 'success', 'result': 30, 'subject': payload['sub']}
    return {'status': 'success', 'result': 'simulated', 'subject': payload['sub']}

valid_token = issue_token('controller-service', ttl_seconds=30)
expired_token = issue_token('controller-service', ttl_seconds=0)
print('VALID:', protected_execute('add 10 and 20', valid_token))
print('EXPIRED:', protected_execute('add 10 and 20', expired_token))
print('INVALID:', protected_execute('add 10 and 20', 'not-a-token'))

## 6) System Flow Explanation

1. Auth service issues token to caller.
2. Caller includes token with request.
3. Protected endpoint verifies signature and expiry.
4. Authorized requests proceed to tool execution.
5. Unauthorized requests return explicit 401 errors.

## 7) Important Concepts You Might Miss

- Signature verification and expiry are separate checks.
- Token subject (`sub`) should map to policy decisions.
- Result endpoints often need auth too, not only execute endpoints.

## 8) Common Mistakes / Pitfalls

- Accepting unsigned or malformed tokens.
- Ignoring expiration in validation path.
- Logging full tokens in plaintext logs.

## 9) Key Takeaways

- JWT-style auth is stronger than static keys for distributed orchestration.
- Validation must cover structure, signature, and expiry.
- Security logic must wrap both execution and retrieval APIs.

## 10) What’s Next (Strict Preview)

`A2A_vs_MCP` compares orchestration styles directly.
It solves conceptual confusion around where A2A coordination ends and MCP tool execution begins.
This matters for choosing the right architecture for complex workloads.